# PF→RH Baseline mit synthetischem Datensatz

Dieses Notebook demonstriert einen vollständigen PF+RH-Durchlauf auf dem synthetischen Beispielstandort in `data/synthetic_site/`.
Es erstellt einen Merge aus den Standard-Konfigurationen, fixiert das PF-Design im Rolling-Horizon-Schritt und fasst Energie- bzw. Kostenbilanzen zusammen.


In [ ]:
from pathlib import Path

from energis.run import rolling_horizon
from energis.run import orchestrator
from energis.io.plotter import export_plots, HAVE_MATPLOTLIB

CONFIG_PATHS = [
    "configs/base.yaml",
    "configs/tech_catalog.yaml",
    "configs/sites/synthetic_example.site.yaml",
    "configs/systems/baseline.system.yaml",
    "configs/scenarios/pf_then_rh.workflow.scenario.yaml",
]

# Workflow ausführen
workflow = rolling_horizon.run_workflow(CONFIG_PATHS)

# NEUE FUNKTION: Vollständiger Export mit orchestrator.run_all()
print("\n📦 Exportiere vollständige Ergebnisse (Excel, JSON, CSV, Plots)...")
export_meta = orchestrator.run_all(CONFIG_PATHS)

print(f"\n✅ Export abgeschlossen:")
print(f"  Verzeichnis: {export_meta['outdir']}")
print(f"  Excel:       {export_meta.get('scenario_xlsx')}")
print(f"  Design-JSON: {export_meta.get('pf_design_json')}")

# Workflow-Info
print(f"\n📊 Workflow-Schritte: {workflow.plan.steps}")
print(f"  PF-Ergebnis:  {'✅' if workflow.pf_result else '❌'}")
print(f"  RH-Ergebnis:  {'✅' if workflow.rh_result else '❌'}")

## Energie- und Kostenbilanzen

Die folgenden Hilfsfunktionen aggregieren die wichtigsten Kennzahlen aus dem Rolling-Horizon-Ergebnis.


In [ ]:
from typing import Dict, List, Mapping

Series = Mapping[str, List[float]]


def _sum(series: Series, key: str) -> float:
    return float(sum(series.get(key, []) or []))

def heat_balance(table, series: Series) -> Dict[str, float]:
    demand = table.data.get("waermebedarf_MWth", [])
    supplies: Dict[str, float] = {}
    for name, values in series.items():
        if name.endswith("_Q_th_MW") or name in {"TES_discharge_MW", "TES_charge_MW"}:
            supplies[name] = float(sum(values))
    return {
        "demand_total_MWh": float(sum(demand)),
        **supplies,
    }


def cost_breakdown(costs: Mapping[str, float]) -> Dict[str, float]:
    numeric = {}
    for key, value in costs.items():
        if isinstance(value, (int, float)):
            numeric[key] = float(value)
    return numeric

result = workflow.rh_result or workflow.pf_result
heat = heat_balance(result.table, result.series) if result else {}
costs = cost_breakdown(result.costs) if result else {}
heat, costs


## Plots exportieren

Die Plot-Funktion nutzt Matplotlib, fällt aber still zurück, falls das Paket nicht installiert ist.


In [ ]:
plot_dir = Path("notebooks/exports/synthetic_pf_rh")
plot_dir.mkdir(parents=True, exist_ok=True)

if workflow.rh_result:
    generated = export_plots(str(plot_dir), workflow.rh_result.table, workflow.rh_result.series, workflow.rh_result.summary)
    print("Matplotlib verfügbar:", HAVE_MATPLOTLIB)
    print("Erzeugte Dateien:")
    for path in generated:
        print(" -", path)
else:
    print("Kein RH-Ergebnis vorhanden")
